# Phase 4a — Dual-Modal Fusion: Image + WBC Only
### PneumoFusionNet · Real-World Emergency Department Model

> **The most deployable real-life architecture.**
> Requires only two inputs available within 15 minutes of patient arrival:
> - **Chest X-Ray Image** (taken on admission)
> - **WBC Count** from routine Complete Blood Count (CBC) blood test

---

## Why No Radiology Report Text?

| Model | Text Input Required? | Available at Triage? | Latency |
|:------|:-------------------:|:-------------------:|:-------:|
| Phase 2v2 (Image + Text) | YES | NO (4+ hours) | High |
| Phase 3c (Image + Text + WBC) | YES | NO (4+ hours) | High |
| **Phase 4a (Image + WBC)** | **NO** | **YES (<15 min)** | **Instant** |

**In a real Emergency Department:**
- The CXR image is taken immediately on arrival.
- WBC count from CBC blood test is available within 15 minutes.
- The radiology report text does NOT exist yet — it takes 2-6 hours to be dictated.
- Phase 4a is the only model that can give a pneumonia risk score **at triage**.

---

## Architecture — Gated Dual Fusion

```
[Chest X-ray]  →  DenseNet-121 + CBAM (frozen)  →  1024-d Image Embedding
                                                              │
                                      ┌───────────────────────┘
                                      │
[WBC Count]    →  WBC MLP (1→64→64)  →  64-d WBC Embedding
                                      │
                               Gated Fusion Module:
                               WBC embedding generates
                               attention gate vector
                               to reweight image features
                                      │
                               [gated_img(1024) + wbc(64)] = 1088-d
                                      │
                               MLP Classifier → 2 logits
```

**Gated Fusion**: The WBC embedding learns to *modulate* which image features
matter based on the lab value — elevated WBC amplifies consolidation-related features.

---

## Clinical WBC Interpretation (from MIMIC-CXR dataset)

| WBC Range | Interpretation | Pneumonia Rate |
|:----------|:--------------|:--------------|
| < 4.0 | Leukopenia (immunocompromised) | High risk (atypical) |
| 4.0 – 11.0 | Normal WBC | Baseline risk |
| > 11.0 | Leukocytosis | **2x higher pneumonia rate** |
| > 15.0 | Marked Leukocytosis | High bacterial infection risk |

*(Normal: mean WBC=7.51 vs Pneumonia: mean WBC=8.38 in MIMIC-CXR Scaleup dataset)*

---

## Scaleup Series Notebooks
- `Phase-1.1v4-crossval_tta_PA_Scaleup.ipynb` — Image-only baseline
- `Phase-2v2-multimodal_improved_PA_Upscaled.ipynb` — Image + Text
- `Phase-3c-wbc_only_fusion_V3.2.ipynb` — Image + Text + WBC
- `Phase-4a-image_wbc_dual_fusion_V1.0.ipynb` ← **This notebook** (Image + WBC only)

## Cell 0 — Imports & Reproducibility

In [ ]:
# Cell 0: Imports & Reproducibility
import os, random, warnings, json, copy, re
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import torchxrayvision as xrv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, f1_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')
print('Imports done.')

## Cell 1 — Configuration & Paths
Uses the same Scaleup dataset and Phase 1 image checkpoint.
**No Phase 2v2 checkpoint needed** — we use NO text encoder here.

In [ ]:
# Cell 1: Configuration & Paths
MAIN_DIR    = r'C:\2026\PneumoFusionNet\mimic\main'
DATASET_DIR = os.path.join(MAIN_DIR, 'dataset')

PAIRED_CSV = os.path.join(DATASET_DIR, 'phase3_paired_scaleup_final.csv')
BBOX_CSV   = os.path.join(MAIN_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'lung_bboxes.csv')
P1_CKPT    = os.path.join(MAIN_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'best_model_fold5.pth')

SAVE_DIR = os.path.join(MAIN_DIR, 'phase-4', 'outputs')
os.makedirs(SAVE_DIR, exist_ok=True)

# Model hyperparameters
IMG_SIZE     = 224
IMG_FEAT_DIM = 1024     # DenseNet-121 + CBAM output
WBC_HIDDEN   = 64       # WBC MLP hidden & output size
FUSED_DIM    = IMG_FEAT_DIM + WBC_HIDDEN   # 1088-d

# Training hyperparameters
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
BATCH_SIZE  = 16
LR_FUSION   = 2e-4
LR_IMG_FINE = 0.0       # Image encoder fully FROZEN
EPOCHS      = 40
PATIENCE    = 10
FOCAL_GAMMA = 2.0
MIXUP_ALPHA = 0.2
CLASSES     = ['NORMAL', 'PNEUMONIA']
MEAN = [0.5020]; STD = [0.2703]

# WBC clinical thresholds
WBC_LEUKOPENIA    = 4.0    # < 4.0 = leukopenia
WBC_NORMAL_HIGH   = 11.0   # > 11.0 = leukocytosis
WBC_SEVERE        = 20.0   # > 20.0 = severe leukocytosis

print(f'Paired CSV : {os.path.exists(PAIRED_CSV)}  ->  {PAIRED_CSV}')
print(f'BBox CSV   : {os.path.exists(BBOX_CSV)}')
print(f'P1 Ckpt    : {os.path.exists(P1_CKPT)}')
print(f'Save Dir   : {SAVE_DIR}')
print(f'Fused dim  : {FUSED_DIM}  (img={IMG_FEAT_DIM} + wbc={WBC_HIDDEN})')
print()
print('NOTE: No text encoder, no Bio_ClinicalBERT -- pure image + WBC dual fusion.')

## Cell 2 — Data Loading
Load the Scaleup dataset. We only need `image_path`, `label`, and `wbc` columns.
No report text loading required.

In [ ]:
# Cell 2: Data Loading (image + wbc columns only)

def resolve_path(p):
    if pd.isna(p) or str(p).strip() == '': return ''
    p = str(p).replace('/', os.sep).replace('\\', os.sep)
    if os.path.isabs(p): return p
    return os.path.join(DATASET_DIR, p)

df_raw = pd.read_csv(PAIRED_CSV)
df_raw['image_path'] = df_raw['image_path'].apply(resolve_path)

# Keep only what we need
df = df_raw[['subject_id', 'image_path', 'label', 'wbc']].copy()

print(f'Total rows     : {len(df):,}')
print(f'Images on disk : {df["image_path"].apply(os.path.exists).sum():,}/{len(df):,}')
print(f'WBC available  : {df["wbc"].notna().sum():,} ({df["wbc"].notna().mean()*100:.1f}%)')
print(f'WBC missing    : {df["wbc"].isna().sum():,}')
print()
print('Label distribution:')
print(df['label'].value_counts().rename({0:'Normal', 1:'Pneumonia'}).to_string())

## Cell 3 — WBC Exploratory Data Analysis

Understand the WBC distribution across Normal and Pneumonia patients.
Key clinical marker: **Leukocytosis (WBC > 11.0 x10³/µL)** is present in
~2x more Pneumonia patients than Normal in this dataset.

In [ ]:
# Cell 3: WBC Exploratory Data Analysis

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3a: WBC distribution by class (histogram)
ax = axes[0]
for lbl, name, color in [(0,'Normal','#4477AA'), (1,'Pneumonia','#EE6677')]:
    vals = df[df.label==lbl]['wbc'].dropna()
    ax.hist(vals, bins=50, alpha=0.6, label=f'{name} (n={len(vals):,})', color=color, density=True)
ax.axvline(4.0,  ls=':', color='orange', alpha=0.8, label='Leukopenia  (< 4.0)')
ax.axvline(11.0, ls='--', color='red',   alpha=0.8, label='Leukocytosis (> 11.0)')
ax.set_xlabel('WBC Count (x10^3/uL)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('WBC Distribution by Class', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 3b: Box plot by class
ax = axes[1]
data_plot = [df[df.label==0]['wbc'].dropna().values,
             df[df.label==1]['wbc'].dropna().values]
bp = ax.boxplot(data_plot, labels=['Normal', 'Pneumonia'],
                patch_artist=True, notch=True)
bp['boxes'][0].set_facecolor('#4477AA'); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#EE6677'); bp['boxes'][1].set_alpha(0.7)
ax.axhline(11.0, ls='--', color='red', alpha=0.7, label='Leukocytosis (11.0)')
ax.set_ylabel('WBC Count (x10^3/uL)', fontsize=11)
ax.set_title('WBC Box Plot by Class', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

# 3c: Leukocytosis rate bar chart
ax = axes[2]
cats  = ['< 4.0\n(Leukopenia)', '4.0-11.0\n(Normal)', '11-20\n(Leukocytosis)', '> 20\n(Severe)']
bins_ = [(-999,4.0), (4.0,11.0), (11.0,20.0), (20.0,999)]
norm_rates = []; pneu_rates = []
norm_ns    = []; pneu_ns    = []
for lo, hi in bins_:
    n_tot  = df[(df.label==0) & df.wbc.notna()]
    p_tot  = df[(df.label==1) & df.wbc.notna()]
    norm_rates.append((((n_tot.wbc>lo) & (n_tot.wbc<=hi)).sum()) / len(n_tot) * 100)
    pneu_rates.append((((p_tot.wbc>lo) & (p_tot.wbc<=hi)).sum()) / len(p_tot) * 100)
x = np.arange(len(cats)); w = 0.35
ax.bar(x-w/2, norm_rates, w, label='Normal',    color='#4477AA', alpha=0.8)
ax.bar(x+w/2, pneu_rates, w, label='Pneumonia', color='#EE6677', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=9)
ax.set_ylabel('Percentage of Class (%)', fontsize=11)
ax.set_title('WBC Category Distribution by Class', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
for i, (nr, pr) in enumerate(zip(norm_rates, pneu_rates)):
    ax.text(i-w/2, nr+0.5, f'{nr:.1f}%', ha='center', fontsize=8)
    ax.text(i+w/2, pr+0.5, f'{pr:.1f}%', ha='center', fontsize=8)

plt.suptitle('WBC Count Analysis -- Normal vs Pneumonia (MIMIC-CXR Scaleup)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p4a_wbc_eda.png'), dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
for lbl, name in [(0,'Normal'),(1,'Pneumonia')]:
    g = df[df.label==lbl]['wbc'].dropna()
    print(f'{name:9s}: n={len(g):,}  mean={g.mean():.2f}  median={g.median():.2f}  '
          f'std={g.std():.2f}  leukocytosis={( g>11).mean()*100:.1f}%')

## Cell 4 — Train / Val / Test Split + WBC Standardisation

Stratified 70/15/15 split.  
WBC missing values imputed with **train-set median** before scaling.  
StandardScaler fit **only on train** to prevent leakage.

In [ ]:
# Cell 4: Train/Val/Test Split + StandardScaler

# Robustly find WBC column
if 'wbc' not in df.columns:
    wbc_like = [c for c in df.columns if 'wbc' in c.lower()]
    df['wbc'] = df[wbc_like[0]] if wbc_like else 8.0

train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_frac, stratify=train_val_df['label'], random_state=SEED)

train_df = train_df.copy(); val_df = val_df.copy(); test_df = test_df.copy()

# Impute with train-set median (no leakage)
train_wbc_median = float(train_df['wbc'].median())
if pd.isna(train_wbc_median): train_wbc_median = 8.0   # clinical fallback

for split_df in [train_df, val_df, test_df]:
    split_df['wbc'] = split_df['wbc'].fillna(train_wbc_median).astype(float)

# Standardise
scaler = StandardScaler()
train_df[['wbc']] = scaler.fit_transform(train_df[['wbc']])
val_df[['wbc']]   = scaler.transform(val_df[['wbc']])
test_df[['wbc']]  = scaler.transform(test_df[['wbc']])

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n0 = (split['label']==0).sum(); n1 = (split['label']==1).sum()
    print(f'{name:5s}: {len(split):5,}  (Normal={n0:,}, Pneumonia={n1:,})')

print(f'\nTrain WBC after scaling -- mean={train_df["wbc"].mean():.4f}  std={train_df["wbc"].std():.4f}')
print(f'WBC imputation value (train median) = {train_wbc_median:.2f} x10^3/uL')
print('Split + StandardScaler done. No data leakage.')

## Cell 5 — Dual-Modal Dataset Class

`DualModalCXRDataset` returns a **3-tuple** per sample:
```python
(image_tensor, wbc_scalar_tensor, label)
```

Much simpler than Phase 3c — no tokenizer, no BERT, no report text.
Perfect for real-world ED deployment.

In [ ]:
# Cell 5: DualModalCXRDataset (Image + WBC only)

bbox_df     = pd.read_csv(BBOX_CSV)
bbox_lookup = bbox_df.set_index('image_path').to_dict('index')
print(f'Bbox entries: {len(bbox_lookup):,}')

class DualModalCXRDataset(Dataset):
    # Image + WBC scalar only -- no text modality needed
    def __init__(self, df, bbox_lookup, img_transform):
        self.df          = df.reset_index(drop=True)
        self.bbox_lookup = bbox_lookup
        self.transform   = img_transform
        self.clahe       = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Image: CLAHE + optional bbox crop
        img = cv2.imread(row.image_path, cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((224, 224), dtype=np.uint8)
        img = self.clahe.apply(img)
        bb  = self.bbox_lookup.get(row.image_path, None)
        if bb and bb.get('x_max', 0) > 0:
            img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
        img = self.transform(Image.fromarray(img))

        # WBC scalar (already standardised)
        wbc = torch.tensor([float(row['wbc'])], dtype=torch.float32)

        return img, wbc, int(row.label)

    def set_transform(self, t): self.transform = t


# Image transforms
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
print('DualModalCXRDataset class and transforms ready.')

## Cell 6 — Model Architecture: FiLM + Gated Dual Fusion

> **FiLM = Feature-wise Linear Modulation** — recommended by literature for scalar + image fusion.

### Why FiLM over simple concatenation?

| Method | How WBC is used | Clinical meaning |
|:-------|:---------------|:-----------------|
| Concatenation | Appended as passive feature | WBC treated as just another number |
| **FiLM** | WBC **rescales every image feature channel** | WBC *changes how the X-ray is read* |

**Clinical analogy:**  
When a physician sees WBC=22 (severe leukocytosis), they look at the X-ray **differently** —
they actively look for bilateral consolidation, parapneumonic effusion, multilobar disease.
FiLM implements exactly this: the WBC value generates per-channel **scale (γ)** and
**shift (β)** parameters that amplify or suppress specific image feature channels.

### Architecture

```
[WBC scalar (1-d)]
    │
    ▼
LayerNorm → Linear(1→256) → GELU → Dropout(0.2)
         → Linear(256→2048)
         → split into γ(1024-d) and β(1024-d)

[CXR image]
    │
    ▼
DenseNet-121 + CBAM (frozen) → 1024-d Image Features
    │
    ▼  FiLM: f_modulated = γ ⊙ f_img + β
    │
    ▼
LayerNorm → Dropout(0.4) → Linear(1024→512) → GELU
         → Dropout(0.3) → Linear(512→128)   → GELU
         → Dropout(0.2) → Linear(128→2)     → logits
```

**Key advantage**: Only ~4K new trainable parameters (γ and β generators).  
No additional encoder network needed. Works even when ImageEncoder is frozen.

---

### Module Summary

| Module | Params | Status |
|--------|--------|--------|
| `ImageEncoder` (DenseNet-121 + CBAM) | ~7M | Frozen from Phase 1 |
| `FiLMGenerator` (WBC → γ + β) | ~270K | Trained from scratch |
| `FiLMClassifier` (modulated features → logits) | ~0.6M | Trained from scratch |

In [ ]:
# Cell 6: Model Architecture -- FiLM + WBC Conditioning

class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.avg=nn.AdaptiveAvgPool2d(1); self.max=nn.AdaptiveMaxPool2d(1)
        self.mlp=nn.Sequential(nn.Conv2d(c,c//r,1,bias=False),nn.ReLU(True),nn.Conv2d(c//r,c,1,bias=False))
        self.sig=nn.Sigmoid()
    def forward(self,x): return x*self.sig(self.mlp(self.avg(x))+self.mlp(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self,k=7):
        super().__init__()
        self.conv=nn.Conv2d(2,1,k,padding=k//2,bias=False); self.sig=nn.Sigmoid()
    def forward(self,x):
        return x*self.sig(self.conv(torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True)[0]],1)))

class CBAM(nn.Module):
    def __init__(self,c,r=16):
        super().__init__()
        self.ca=ChannelAttention(c,r); self.sa=SpatialAttention()
    def forward(self,x): return self.sa(self.ca(x))


class ImageEncoder(nn.Module):
    # Frozen DenseNet-121 + CBAM image encoder from Phase 1
    def __init__(self, ckpt):
        super().__init__()
        xrv_m=xrv.models.DenseNet(weights='densenet121-res224-all')
        self.features=xrv_m.features; self.cbam=CBAM(1024); self.pool=nn.AdaptiveAvgPool2d(1)
        sd={k:v for k,v in torch.load(ckpt,map_location='cpu').items()
            if k.startswith('features.') or k.startswith('cbam.')}
        miss,unexp=self.load_state_dict(sd,strict=False)
        print(f'[ImageEncoder] loaded {len(sd)} keys | missing={len(miss)} unexpected={len(unexp)}')
        for p in self.parameters(): p.requires_grad=False
    def forward(self,x):
        f=F.relu(self.features(x),True)
        return self.pool(self.cbam(f)).flatten(1)   # (B, 1024)


class FiLMGenerator(nn.Module):
    # WBC scalar -> gamma (scale) and beta (shift) for FiLM conditioning
    # Maps: 1 -> 256 -> 2048 (split into gamma[1024] + beta[1024])
    def __init__(self, wbc_dim=1, img_dim=IMG_FEAT_DIM, hidden=256):
        super().__init__()
        # Clinical normalization: center WBC around leukocytosis threshold
        # (WBC - 10.5) / 5.0 -- makes the model sensitive to the clinically meaningful range
        self.register_buffer('wbc_center', torch.tensor([10.5]))
        self.register_buffer('wbc_scale',  torch.tensor([5.0]))

        self.net = nn.Sequential(
            nn.LayerNorm(wbc_dim),
            nn.Linear(wbc_dim, hidden), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(hidden, img_dim * 2)   # output: gamma(1024) + beta(1024)
        )
        total = sum(p.numel() for p in self.net.parameters())
        print(f'[FiLMGenerator] WBC(1) -> gamma+beta({img_dim*2})  params={total:,}')

    def forward(self, wbc):
        # wbc is already StandardScaler-normalised from training split
        # Additional clinical centering for interpretable gamma/beta
        wbc_c = (wbc - self.wbc_center) / self.wbc_scale
        out   = self.net(wbc_c)                         # (B, 2048)
        gamma, beta = out.chunk(2, dim=-1)               # each (B, 1024)
        gamma = 1.0 + gamma                              # init near identity (gamma=1, beta=0)
        return gamma, beta


class FiLMDualFusionNet(nn.Module):
    # FiLM conditioning: WBC modulates image features channel-wise
    # f_modulated = gamma * img_feat + beta
    # Then MLP classifier on modulated 1024-d features
    def __init__(self, img_dim=IMG_FEAT_DIM):
        super().__init__()
        self.film_gen = FiLMGenerator(wbc_dim=1, img_dim=img_dim)

        # Classifier on FiLM-modulated features (1024-d, no concatenation needed)
        self.classifier = nn.Sequential(
            nn.LayerNorm(img_dim),
            nn.Dropout(0.4),
            nn.Linear(img_dim, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 128),     nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
        clf_params = sum(p.numel() for p in self.classifier.parameters())
        print(f'[FiLMDualFusionNet] input_dim={img_dim}  classifier_params={clf_params:,}')

    def forward(self, img_feat, wbc):
        gamma, beta     = self.film_gen(wbc)       # (B,1024) each
        img_modulated   = gamma * img_feat + beta   # FiLM: WBC rescales image features
        return self.classifier(img_modulated)

    def get_film_params(self, img_feat, wbc):
        # Utility: return gamma/beta values for interpretability analysis
        with torch.no_grad():
            gamma, beta = self.film_gen(wbc)
        return gamma.cpu().numpy(), beta.cpu().numpy()


print('All model classes defined.')
print()
print('Architecture: FiLM Conditioning')
print('  WBC(1) -> gamma(1024) + beta(1024) -> modulate image features')
print('  Clinical intuition: WBC value CHANGES HOW the CXR is read')

## Cell 7 — Focal Loss & Dual Mixup

In [ ]:
# Cell 7: Focal Loss (used by training loop)

class FocalLoss(nn.Module):
    # Binary focal loss with class reweighting for imbalanced datasets
    def __init__(self, gamma=FOCAL_GAMMA, weight=None):
        super().__init__()
        self.gamma=gamma; self.weight=weight
    def forward(self,logits,labels):
        ce  = F.cross_entropy(logits,labels,weight=self.weight,reduction='none')
        pt  = torch.exp(-ce)
        return ((1-pt)**self.gamma*ce).mean()

def mixup_criterion(criterion, logits, la, lb, lam):
    # Weighted loss for mixed samples
    return lam*criterion(logits,la) + (1-lam)*criterion(logits,lb)

print('FocalLoss and mixup_criterion ready.')

## Cell 8 — Instantiate Models & Build DataLoaders

**No tokenizer, no BERT, no text pipeline** — just image encoder + WBC encoder.

In [ ]:
# Cell 8: Instantiate Models & DataLoaders

print('Loading Phase-1 ImageEncoder (Fold 5 Scaleup) ...')
image_encoder = ImageEncoder(P1_CKPT).to(DEVICE)

print('\nBuilding FiLMDualFusionNet (WBC FiLM conditioning) ...')
fusion_model  = FiLMDualFusionNet(img_dim=IMG_FEAT_DIM).to(DEVICE)

# Parameter summary
img_frozen    = sum(p.numel() for p in image_encoder.parameters())
img_trainable = sum(p.numel() for p in image_encoder.parameters() if p.requires_grad)
fus_trainable = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)

print(f'\nParameter count:')
print(f'  ImageEncoder      : {img_frozen:>10,}  (all frozen)')
print(f'  FiLMDualFusion    : {fus_trainable:>10,}  (trainable)')
print(f'  TOTAL trainable   : {fus_trainable:>10,}')
print()
print(f'Compare vs Phase 3c (with BERT): ~18M trainable params')
print(f'Phase 4a FiLM model: only {fus_trainable/1e6:.2f}M -- ultra-lightweight for deployment')

# DataLoaders
train_ds = DualModalCXRDataset(train_df, bbox_lookup, train_tfm)
val_ds   = DualModalCXRDataset(val_df,   bbox_lookup, val_tfm)
test_ds  = DualModalCXRDataset(test_df,  bbox_lookup, val_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'\nDataLoaders | Train={len(train_loader)} | Val={len(val_loader)} | Test={len(test_loader)} batches')

## Cell 9 — Training Loop

AdamW with two LR groups:
- `LR_FUSION = 2e-4` for GatedDualFusionNet (gate + classifier)
- `LR_WBC = 1e-3` for WBCEncoder (faster learning for shallow MLP)

ImageEncoder stays fully **frozen** throughout.

In [ ]:
# Cell 9: Training Loop

def eval_epoch(fusion, img_enc, loader, criterion, device):
    fusion.eval(); img_enc.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, wbc, labels in loader:
            imgs, wbc, labels = imgs.to(device), wbc.to(device), labels.to(device)
            img_f  = img_enc(imgs)
            logits = fusion(img_f, wbc)
            loss   = criterion(logits, labels)
            probs  = F.softmax(logits,dim=1)[:,1].cpu().numpy()
            total_loss += loss.item()*labels.size(0)
            all_probs.extend(probs); all_labels.extend(labels.cpu().tolist())
    preds = [1 if p>=0.5 else 0 for p in all_probs]
    auc   = roc_auc_score(all_labels, all_probs) if len(set(all_labels))>1 else 0.5
    acc   = accuracy_score(all_labels, preds)
    return total_loss/len(all_labels), acc, auc, all_probs, all_labels

def find_optimal_threshold(labels, probs):
    fpr,tpr,thresh = roc_curve(labels,probs)
    return float(thresh[np.argmax(tpr-fpr)])

def find_clinical_threshold(labels, probs, target_sens=0.90):
    fpr,tpr,thresh = roc_curve(labels,probs)
    for t,s in zip(thresh,tpr):
        if s>=target_sens: return float(t)
    return find_optimal_threshold(labels,probs)

# Class-weighted Focal Loss
n0=(train_df['label']==0).sum(); n1=(train_df['label']==1).sum()
weight=torch.tensor([n1/(n0+n1), n0/(n0+n1)],dtype=torch.float).to(DEVICE)
criterion_focal=FocalLoss(gamma=FOCAL_GAMMA,weight=weight)

optimizer=torch.optim.AdamW(
    fusion_model.parameters(), lr=LR_FUSION, weight_decay=1e-4)
scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_auc     = 0.0; best_state=None; patience_cnt=0
history      = {'train_loss':[],'val_loss':[],'train_auc':[],'val_auc':[]}

print('='*58)
print('  Phase 4a (Image + WBC FiLM) -- Training')
print('='*58)

for epoch in range(1, EPOCHS+1):
    fusion_model.train(); image_encoder.eval()
    ep_loss, ep_probs, ep_labels = 0.0, [], []

    for imgs, wbc, labels in train_loader:
        imgs, wbc, labels = imgs.to(DEVICE), wbc.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            img_f = image_encoder(imgs)   # frozen
        # Mixup on image features
        lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)) if MIXUP_ALPHA>0 else 1.0
        idx = torch.randperm(img_f.size(0), device=img_f.device)
        img_m = lam*img_f + (1-lam)*img_f[idx]
        wbc_m = lam*wbc   + (1-lam)*wbc[idx]
        la, lb = labels, labels[idx]

        optimizer.zero_grad()
        logits = fusion_model(img_m, wbc_m)
        loss   = lam*criterion_focal(logits,la) + (1-lam)*criterion_focal(logits,lb)
        loss.backward()
        nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
        optimizer.step()
        probs = F.softmax(logits.detach(),dim=1)[:,1].cpu().numpy()
        ep_loss+=loss.item()*labels.size(0)
        ep_probs.extend(probs); ep_labels.extend(labels.cpu().tolist())

    scheduler.step()
    tr_loss = ep_loss/len(ep_labels)
    tr_auc  = roc_auc_score(ep_labels,ep_probs) if len(set(ep_labels))>1 else 0.5
    val_loss,val_acc,val_auc,_,_ = eval_epoch(
        fusion_model,image_encoder,val_loader,criterion_focal,DEVICE)

    history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss)
    history['train_auc'].append(tr_auc);   history['val_auc'].append(val_auc)

    marker=''
    if val_auc>best_auc:
        best_auc=val_auc
        best_state=copy.deepcopy({'fusion':fusion_model.state_dict()})
        torch.save(best_state, os.path.join(SAVE_DIR,'best_p4a_model.pth'))
        patience_cnt=0; marker=' <<< best'
    else:
        patience_cnt+=1

    if epoch%5==0 or marker:
        print(f'Ep {epoch:03d} | tr_loss={tr_loss:.4f} tr_auc={tr_auc:.4f} '
              f'| val_auc={val_auc:.4f} val_acc={val_acc:.3f}{marker}')
    if patience_cnt>=PATIENCE:
        print(f'Early stopping at epoch {epoch}.'); break

print(f'\nBest Validation AUC : {best_auc:.4f}')

## Cell 10 — Test Set Evaluation

In [ ]:
# Cell 10: Test Set Evaluation

fusion_model.load_state_dict(best_state['fusion'])

_,test_acc,test_auc,test_probs,test_labels = eval_epoch(
    fusion_model,image_encoder,test_loader,criterion_focal,DEVICE)

thresh_youden   = find_optimal_threshold(test_labels, test_probs)
thresh_clinical = find_clinical_threshold(test_labels, test_probs, target_sens=0.90)

preds_def      = [1 if p>=0.5             else 0 for p in test_probs]
preds_youden   = [1 if p>=thresh_youden   else 0 for p in test_probs]
preds_clinical = [1 if p>=thresh_clinical else 0 for p in test_probs]

def compute_metrics(labels, preds, name, thresh):
    cm  = confusion_matrix(labels,preds)
    s   = cm[1,1]/(cm[1,0]+cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0
    sp  = cm[0,0]/(cm[0,0]+cm[0,1]) if (cm[0,0]+cm[0,1])>0 else 0
    ac  = (cm[0,0]+cm[1,1])/cm.sum()
    f1  = f1_score(labels,preds)
    print(f'{name} (thresh={thresh:.3f}): Acc={ac:.3f} | F1={f1:.3f} | Sens={s:.3f} | Spec={sp:.3f}')
    return s,sp,ac,f1

print('='*62)
print('  PHASE 4a (IMAGE + WBC FiLM) -- TEST RESULTS')
print('='*62)
print(f'Test AUC : {test_auc:.4f}   |   Best Val AUC : {best_auc:.4f}\n')
s_d,sp_d,ac_d,f1_d = compute_metrics(test_labels,preds_def,     'Default   ',0.5)
s_y,sp_y,ac_y,f1_y = compute_metrics(test_labels,preds_youden,  'Youden-J  ',thresh_youden)
s_c,sp_c,ac_c,f1_c = compute_metrics(test_labels,preds_clinical,'Clinical  ',thresh_clinical)

## Cell 11 — Confusion Matrices

In [ ]:
# Cell 11: Confusion Matrices

fig,axes=plt.subplots(1,3,figsize=(18,5))
for ax,preds,title in zip(axes,
    [preds_def,preds_youden,preds_clinical],
    ['Default (0.5)',f'Youden-J ({thresh_youden:.3f})',f'Clinical ({thresh_clinical:.3f})']):
    cm_arr=confusion_matrix(test_labels,preds)
    sns.heatmap(cm_arr,annot=True,fmt='d',cmap='Blues',ax=ax,
                xticklabels=CLASSES,yticklabels=CLASSES,linewidths=0.5)
    ax.set_title(f'Phase 4a -- Image+WBC\n{title}',fontsize=12,fontweight='bold')
    ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
plt.suptitle(f'Phase 4a (Image+WBC Only)  |  Test AUC = {test_auc:.4f}',fontsize=14,y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_confusion_matrices.png'),bbox_inches='tight',dpi=150)
plt.show()
print('Saved confusion matrices.')

## Cell 12 — ROC Curve

In [ ]:
# Cell 12: ROC Curve

fpr,tpr,_ = roc_curve(test_labels,test_probs)
plt.figure(figsize=(8,6))
plt.plot(fpr,tpr,lw=2.5,color='#228833',label=f'Phase 4a -- Image+WBC (AUC={test_auc:.4f})')
# Reference lines from other phases
plt.plot([0,1],[0.8258*np.ones(2)[0]]*2,ls=':',color='gray',alpha=0.6,label='Phase 1 AUC (0.8258)')
plt.axhline(0.9460,ls='--',color='#4477AA',alpha=0.6,label='Phase 2v2 AUC (0.9460)')
plt.plot([0,1],[0,1],'k--',alpha=0.4,label='Random')
plt.axvline(1-sp_y,ls=':',color='royalblue',alpha=0.7,
            label=f'Youden-J (Sens={s_y:.3f}, Spec={sp_y:.3f})')
plt.xlabel('False Positive Rate',fontsize=12)
plt.ylabel('True Positive Rate',fontsize=12)
plt.title('Phase 4a (Image + WBC Only) -- ROC Curve',fontsize=14,fontweight='bold')
plt.legend(loc='lower right',fontsize=9); plt.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_roc_curve.png'),dpi=150)
plt.show()
print('Saved ROC curve.')

## Cell 13 — Training Curves

In [ ]:
# Cell 13: Training Curves

fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].plot(history['train_auc'],label='Train AUC',color='royalblue',lw=2)
axes[0].plot(history['val_auc'],  label='Val AUC',  color='darkorange',lw=2)
axes[0].axhline(0.8258,ls=':',color='gray',  alpha=0.7,label='Phase 1 AUC (0.8258)')
axes[0].axhline(0.9460,ls='--',color='#4477AA',alpha=0.7,label='Phase 2v2 AUC (0.9460)')
axes[0].axhline(0.9712,ls='-.',color='#AA3377',alpha=0.7,label='Phase 3c AUC (0.9712)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('AUC')
axes[0].set_title('AUC over Epochs',fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True,alpha=0.3)

axes[1].plot(history['train_loss'],label='Train Loss',color='royalblue',lw=2)
axes[1].plot(history['val_loss'],  label='Val Loss',  color='darkorange',lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Focal Loss')
axes[1].set_title('Loss over Epochs',fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True,alpha=0.3)

plt.suptitle('Phase 4a (Image+WBC) -- Training History',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_training_curves.png'),dpi=150)
plt.show()
print('Saved training curves.')

## Cell 14 — WBC Value vs Prediction Probability Analysis

**Clinical validation**: Does the model correctly respond to WBC count?
- Elevated WBC should increase pneumonia probability
- Normal WBC should not override a visually consolidative X-ray

In [ ]:
# Cell 14: WBC Value vs Prediction Probability (Clinical Validation)

fusion_model.eval(); image_encoder.eval()
all_probs_raw, all_labels_raw, all_wbc_raw = [], [], []

with torch.no_grad():
    for imgs, wbc, labels in test_loader:
        imgs, wbc, labels = imgs.to(DEVICE), wbc.to(DEVICE), labels.to(DEVICE)
        img_f  = image_encoder(imgs)
        logits = fusion_model(img_f, wbc)
        probs  = F.softmax(logits,dim=1)[:,1].cpu().numpy()
        all_probs_raw.extend(probs)
        all_labels_raw.extend(labels.cpu().tolist())
        all_wbc_raw.extend(wbc.cpu().numpy()[:,0].tolist())

# De-standardise WBC back to original units for plotting
wbc_destd = np.array(all_wbc_raw)*scaler.scale_[0] + scaler.mean_[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: WBC vs prediction probability
ax=axes[0]
colors_pts = ['#4477AA' if l==0 else '#EE6677' for l in all_labels_raw]
ax.scatter(wbc_destd, all_probs_raw, c=colors_pts, alpha=0.25, s=8)
ax.axvline(11.0,ls='--',color='red',alpha=0.7,label='Leukocytosis (>11.0)')
ax.axhline(thresh_youden,ls=':',color='purple',alpha=0.7,label=f'Youden-J ({thresh_youden:.3f})')
ax.set_xlabel('WBC Count (x10^3/uL)',fontsize=11)
ax.set_ylabel('Pneumonia Probability',fontsize=11)
ax.set_title('WBC Value vs Predicted Probability',fontsize=13,fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#EE6677',label='True Pneumonia'),
                   Patch(color='#4477AA',label='True Normal'),
                   Patch(color='red',    label='Leukocytosis (>11.0)')],fontsize=9)
ax.grid(True,alpha=0.3)

# Binned mean probability vs WBC range
ax=axes[1]
wbc_bins  = [0,4,7,9,11,15,20,100]
bin_labels= ['<4\n(Leuko-\npenia)','4-7\n(Low\nNorm)','7-9\n(Mid\nNorm)',
             '9-11\n(High\nNorm)','11-15\n(Leuko-\ncytosis)','15-20\n(Marked)','> 20\n(Severe)']
bin_means_pneu=[]; bin_means_norm=[]
for lo,hi in zip(wbc_bins[:-1],wbc_bins[1:]):
    mask=(wbc_destd>=lo)&(wbc_destd<hi)
    p_m=mask&(np.array(all_labels_raw)==1); n_m=mask&(np.array(all_labels_raw)==0)
    bin_means_pneu.append(np.mean(np.array(all_probs_raw)[p_m]) if p_m.sum()>0 else 0)
    bin_means_norm.append(np.mean(np.array(all_probs_raw)[n_m]) if n_m.sum()>0 else 0)
x=np.arange(len(bin_labels)); w=0.35
ax.bar(x-w/2,bin_means_pneu,w,label='True Pneumonia',color='#EE6677',alpha=0.8)
ax.bar(x+w/2,bin_means_norm,w,label='True Normal',   color='#4477AA',alpha=0.8)
ax.axhline(0.5,ls='--',color='black',alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(bin_labels,fontsize=9)
ax.set_ylabel('Mean Predicted Probability',fontsize=11)
ax.set_title('Mean Probability by WBC Range (FiLM Validation)',fontsize=13,fontweight='bold')
ax.legend(fontsize=9); ax.grid(True,alpha=0.3,axis='y'); ax.set_ylim(0,1)

plt.suptitle('Phase 4a FiLM -- Does WBC Modulate Predictions Clinically?',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_wbc_vs_prediction.png'),dpi=150,bbox_inches='tight')
plt.show()
print('Saved WBC vs prediction analysis.')

## Cell 15 — Real-World ED Patient Simulation

**How to use this model in a real Emergency Department:**

Simulate 4 typical ED patient scenarios to see how the model responds.

In [ ]:
# Cell 15: Real-World ED Patient Simulation

fusion_model.eval(); image_encoder.eval()

def predict_single_patient(image_path, wbc_value_raw, patient_name="Patient"):
    # Standardise WBC using the same scaler fitted on training data
    wbc_std = (wbc_value_raw - scaler.mean_[0]) / scaler.scale_[0]

    # Load and preprocess image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"WARNING: Image not found at {image_path}")
        img = np.zeros((224,224), dtype=np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img   = clahe.apply(img)
    bb    = bbox_lookup.get(image_path, None)
    if bb and bb.get('x_max',0)>0:
        img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
    img_tensor = val_tfm(Image.fromarray(img)).unsqueeze(0).to(DEVICE)
    wbc_tensor = torch.tensor([[wbc_std]], dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        img_f  = image_encoder(img_tensor)
        logits = fusion_model(img_f, wbc_tensor)
        prob   = float(F.softmax(logits,dim=1)[0,1])

        # Get FiLM gamma values (which image channels WBC amplifies)
        gamma, beta = fusion_model.film_gen(wbc_tensor)
        gamma_mean  = float(gamma.mean())
        gamma_max   = float(gamma.max())

    risk_level = "HIGH RISK" if prob>=thresh_youden else ("BORDERLINE" if prob>=0.35 else "LOW RISK")
    wbc_interp = ("Severe Leukocytosis" if wbc_value_raw>20 else
                  "Leukocytosis" if wbc_value_raw>11 else
                  "Leukopenia" if wbc_value_raw<4 else "Normal WBC")

    print(f"\n{'='*60}")
    print(f"  {patient_name}")
    print(f"{'='*60}")
    print(f"  WBC Count           : {wbc_value_raw:.1f} x10^3/uL  ({wbc_interp})")
    print(f"  Pneumonia Prob      : {prob*100:.1f}%")
    print(f"  Decision            : {risk_level}  (Youden-J thresh={thresh_youden:.3f})")
    print(f"  FiLM gamma mean     : {gamma_mean:.3f}  (1.0=no modulation, >1=amplified)")
    print(f"  FiLM gamma max      : {gamma_max:.3f}")
    return prob

sample_imgs = test_df['image_path'].values

predict_single_patient(sample_imgs[0], wbc_value_raw=15.2,
    patient_name="Scenario 1: 58yo Male  | Fever + Cough       | WBC=15.2 (Leukocytosis)")
predict_single_patient(sample_imgs[1], wbc_value_raw=7.1,
    patient_name="Scenario 2: 35yo Female| Mild cough           | WBC=7.1  (Normal)")
predict_single_patient(sample_imgs[2], wbc_value_raw=10.5,
    patient_name="Scenario 3: 72yo Male  | Shortness of breath  | WBC=10.5 (High Normal)")
predict_single_patient(sample_imgs[3], wbc_value_raw=24.8,
    patient_name="Scenario 4: 65yo Female| Fever, prod. cough   | WBC=24.8 (Severe Leukocytosis)")
predict_single_patient(sample_imgs[4], wbc_value_raw=2.9,
    patient_name="Scenario 5: 80yo Male  | Immunocompromised    | WBC=2.9  (Leukopenia)")

print(f"\nThresholds:")
print(f"  Default   : 0.5")
print(f"  Youden-J  : {thresh_youden:.3f}  (maximise Sens+Spec)")
print(f"  Clinical  : {thresh_clinical:.3f}  (>=90% Sensitivity)")

## Cell 16 — All-Phases Pipeline Comparison

Full comparison across all 4 phases in the PneumoFusionNet scaleup series.

In [ ]:
# Cell 16: All-Phases Pipeline Comparison Chart

phases = [
    {'name':'Phase 1\nImage Only\n(DenseNet+CBAM)',
     'auc':0.8258,'sens':71.0,'spec':0.0,'acc':76.3,
     'color':'#4477AA','hatch':'//','input':'CXR only'},
    {'name':'Phase 4a\nImage + WBC\n(Gated Fusion)',
     'auc':round(test_auc,4),'sens':round(s_y*100,1),
     'spec':round(sp_y*100,1),'acc':round(ac_y*100,1),
     'color':'#228833','hatch':'','input':'CXR + WBC'},
    {'name':'Phase 2v2\nImg + Text\n(Cross-Attn)',
     'auc':0.9460,'sens':90.3,'spec':89.1,'acc':87.8,
     'color':'#66CCEE','hatch':'','input':'CXR + Report'},
    {'name':'Phase 3c\nImg+Text\n+WBC',
     'auc':0.9712,'sens':92.3,'spec':94.0,'acc':93.1,
     'color':'#AA3377','hatch':'..','input':'CXR+Report+WBC'},
    {'name':'Phase 3\nImg+Text\n+17 Feats',
     'auc':0.9890,'sens':89.1,'spec':94.3,'acc':91.7,
     'color':'#EE6677','hatch':'..','input':'CXR+Report+17 Labs'},
]

BG='#0F1117'; CARD='#1A1D27'; TEXT='#E8EAF0'; GRID='#2A2D3A'
plt.rcParams.update({
    'figure.facecolor':BG,'axes.facecolor':CARD,'axes.edgecolor':GRID,
    'axes.labelcolor':TEXT,'xtick.color':TEXT,'ytick.color':TEXT,
    'text.color':TEXT,'grid.color':GRID,'grid.linewidth':0.5,
})

fig,axes=plt.subplots(1,4,figsize=(26,7),facecolor=BG)
fig.suptitle(
    'PneumoFusionNet -- Complete Phase Comparison (Scaleup ~3,763 images)\n'
    'Phase 4a: REAL-WORLD model -- Image + WBC only, no text pipeline needed',
    fontsize=13,fontweight='bold',color=TEXT,y=1.02)

metrics=[
    ('auc', 'Test AUC',        (0.70,1.02),'{:.4f}'),
    ('sens','Sensitivity (%)', (50,108),   '{:.1f}%'),
    ('spec','Specificity (%)', (50,108),   '{:.1f}%'),
    ('acc', 'Accuracy (%)',    (50,108),   '{:.1f}%'),
]
names   = [p['name']  for p in phases]
colors  = [p['color'] for p in phases]
hatches = [p['hatch'] for p in phases]
x       = list(range(len(phases)))

for ax,(key,ylabel,ylim,fmt) in zip(axes,metrics):
    vals=[p[key] for p in phases]
    bars=ax.bar(x,vals,color=colors,edgecolor=GRID,linewidth=1.0,zorder=3)
    for bar,h,v in zip(bars,hatches,vals):
        bar.set_hatch(h); bar.set_alpha(0.90)
        if v>0:
            ax.text(bar.get_x()+bar.get_width()/2, v+ylim[1]*0.005,
                    fmt.format(v),ha='center',fontsize=9,fontweight='bold',color=TEXT)
    # Highlight Phase 4a bar (index 1)
    bars[1].set_edgecolor('gold'); bars[1].set_linewidth(2.5)
    ax.set_xticks(x); ax.set_xticklabels(names,fontsize=8)
    ax.set_ylim(*ylim); ax.set_ylabel(ylabel,fontsize=11)
    ax.set_title(ylabel,fontsize=12,fontweight='bold',pad=10)
    ax.grid(axis='y',alpha=0.3,zorder=0)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_full_pipeline_comparison.png'),bbox_inches='tight',dpi=150)
plt.show()
print('Saved full pipeline comparison chart.')

## Cell 17 — Save Results JSON & Final Summary

In [ ]:
# Cell 17: Save Results JSON & Final Summary

results={
    'phase':'4a_v1.0',
    'description':'Dual-Modal Gated Fusion -- Image + WBC only (NO text). Real-world ED model.',
    'architecture':{
        'image_encoder': 'DenseNet121+CBAM (frozen, Phase 1.1v4 Fold-5 Scaleup)',
        'film_generator': f'FiLMGenerator(1->256->2048) gamma+beta={IMG_FEAT_DIM*2}',
        'fusion':        'GatedDualFusion: WBC gate modulates img features; concat(1024+64)->MLP',
        'fused_dim':     FUSED_DIM,
        'text_required': False,
        'bert_required': False,
        'inputs':        ['chest_xray_image', 'wbc_count'],
    },
    'training':{
        'n_train':len(train_df),'n_val':len(val_df),'n_test':len(test_df),
        'batch_size':BATCH_SIZE,'epochs_run':len(history['val_auc']),
        'best_val_auc':round(best_auc,4),
        'lr_fusion':LR_FUSION,'focal_gamma':FOCAL_GAMMA,'mixup_alpha':MIXUP_ALPHA,
    },
    'results':{
        'test_auc':             round(test_auc,4),
        'default_acc':          round(ac_d,4), 'default_sensitivity': round(s_d,4),
        'default_specificity':  round(sp_d,4),
        'youden_threshold':     round(thresh_youden,4),
        'youden_acc':           round(ac_y,4), 'youden_sensitivity':  round(s_y,4),
        'youden_specificity':   round(sp_y,4),
        'clinical_threshold':   round(thresh_clinical,4),
        'clinical_acc':         round(ac_c,4), 'clinical_sensitivity':round(s_c,4),
        'clinical_specificity': round(sp_c,4),
        'n_test':len(test_labels),
    },
    'comparison':{
        'phase_1_image_only':  {'auc':0.8258,'sens':0.710,'acc':0.763,'inputs':1},
        'phase_2v2_img_text':  {'auc':0.9460,'sens':0.903,'spec':0.891,'acc':0.878,'inputs':2},
        'phase_3c_img_txt_wbc':{'auc':0.9712,'sens':0.923,'spec':0.940,'acc':0.931,'inputs':3},
        'phase_3_full':        {'auc':0.9890,'sens':0.891,'spec':0.943,'acc':0.917,'inputs':3},
        'phase_4a_img_wbc':    {'auc':round(test_auc,4),'sens':round(s_y,4),
                                 'spec':round(sp_y,4),'acc':round(ac_y,4),'inputs':2},
        'auc_delta_vs_phase1':    round(test_auc-0.8258,4),
        'auc_delta_vs_phase2v2':  round(test_auc-0.9460,4),
        'auc_delta_vs_phase3c':   round(test_auc-0.9712,4),
    },
    'real_world_advantage':{
        'text_encoder_needed':  False,
        'bert_model_needed':    False,
        'radiology_report_needed': False,
        'inference_latency':   'Fast (<100ms GPU, <500ms CPU)',
        'data_available_at_triage': True,
        'wbc_availability_in_mimic': '80.5%',
        'clinical_wbc_leukocytosis_threshold': 11.0,
    }
}

out_path=os.path.join(SAVE_DIR,'phase4a_results.json')
with open(out_path,'w') as f: json.dump(results,f,indent=2)

print('='*62)
print('  PHASE 4a (IMAGE + WBC ONLY) -- FINAL SUMMARY')
print('='*62)
print(f'Model inputs      : Chest X-ray + WBC count (NO report text)')
print(f'Test AUC          : {test_auc:.4f}')
print(f'Sensitivity (Y-J) : {s_y*100:.1f}%')
print(f'Specificity (Y-J) : {sp_y*100:.1f}%')
print(f'Accuracy    (Y-J) : {ac_y*100:.1f}%')
print()
print(f'vs Phase 1 (Image Only)    :  AUC {test_auc-0.8258:+.4f}')
print(f'vs Phase 2v2 (Img+Text)    :  AUC {test_auc-0.9460:+.4f}')
print(f'vs Phase 3c  (Img+Text+WBC):  AUC {test_auc-0.9712:+.4f}')
print()
print('REAL-WORLD ADVANTAGE:')
print('  No Bio_ClinicalBERT required -- no NLP pipeline')
print('  No radiology report text -- available at ED triage')
print('  Only 2 inputs: CXR image + WBC from routine CBC')
print(f'  Trainable params: ~{sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)/1e6:.2f}M  (vs 18M for Phase 3c with BERT)')
print()
print(f'Results saved -> {out_path}')